[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ImagingDataCommons/CloudSegmentator/blob/main/workflows/MOOSE/Notebooks/moosePostProcessNotebook.ipynb)

# **MOOSE Post-Processing (Task 2): Convert NIfTI segmentations to DICOM-SEG**

This notebook is the CPU-side of the MOOSE twoVM workflow. It:
1. Extracts the `moose_segmentations.tar.lz4` archive produced by Task 1
2. Re-downloads source DICOM from IDC for each series (needed by dcmqi as reference)
3. For each series/model pair, builds a dcmqi config JSON and runs `itkimage2segimage`
4. Tars + lz4-compresses all DICOM-SEG files as `moose_dicom_seg.tar.lz4`

Please cite:

Herz C, Fillion-Robin JC, Onken M, Riesmeier J, Lasso A, Pinter C, Fichtinger G, Pieper S, Clunie D, Kikinis R, Fedorov A. dcmqi: An Open Source Library for Standardized Communication of Quantitative Image Analysis Results Using DICOM. Cancer Res. 2017 Nov 1;77(21):e87-e90. https://doi.org/10.1158/0008-5472.CAN-17-0336

Shiyam Sundar LK, et al. Fully automated, semantic segmentation of whole-body 18F-FDG PET/CT images based on data-centric artificial intelligence. J Nucl Med. 2022. https://doi.org/10.2967/jnumed.122.264063

## Setup / prerequisites

This notebook is normally run inside the IDC `post_process_moose` Docker image (via Terra + papermill), where all prerequisites are pre-installed. The cell below makes it **also runnable standalone** — in Google Colab or any local Jupyter / Python kernel — by installing the missing pieces on demand:

* Python packages (`nibabel`, `numpy<2`, `pydicom`, `idc-index`, …)
* the `lz4` CLI (de/compress the `.tar.lz4` archives)
* [dcmqi](https://github.com/QIICR/dcmqi) — provides `itkimage2segimage` with `--segmentationType labelmap` support. Installed via the PyPI [`dcmqi`](https://github.com/ImagingDataCommons/dcmqi-python-distributions) package, which bundles the compiled binaries; package `0.4.1` ships dcmqi binaries **v1.5.5** (the package version is independent of the dcmqi release version).
* MOOSE's `SNOMED.py` label-mapping module

Every step is gated on a presence check, so inside the Docker image (and on re-runs) it is a no-op. To try it out, point `segmentationArchivePath` (in the Parameters cell) at a `moose_segmentations.tar.lz4` produced by Task 1.

In [ ]:
# --- Prerequisite installation (gated) ---------------------------------------
# Inside the IDC `post_process_moose` Docker image (Terra / papermill execution)
# every prerequisite below is already baked in, so each check is a no-op and
# nothing is installed or downloaded. When this notebook is run standalone --
# in Google Colab or any local Jupyter / Python kernel -- the missing pieces are
# installed on the fly so the rest of the notebook runs unchanged.
#
# Mirrors the relevant parts of
#   workflows/MOOSE/Dockerfiles/post_process_moose/Dockerfile
# (the post-process step only needs lz4, dcmqi's itkimage2segimage and a handful
# of Python packages -- it does not use dcm2niix / s5cmd / pigz).
import importlib.util
import os
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def _has_module(name: str) -> bool:
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False


# 1. Python packages. import-name -> pip spec (matching the Dockerfile pins).
_required_pkgs = {
    "nibabel": "nibabel==5.1.0",
    "numpy": "numpy<2",
    "pandas": "pandas==1.5.3",
    "pydicom": "pydicom==2.4.1",
    "requests": "requests==2.31.0",
    "tqdm": "tqdm==4.65.0",
    "idc_index": "idc-index",
    # Only needed for the optional GCS upload / DICOM-store import cells, but
    # cheap to ensure and usually already present on Colab.
    "google.cloud.storage": "google-cloud-storage==2.16.0",
}
_missing = sorted({spec for mod, spec in _required_pkgs.items() if not _has_module(mod)})
if _missing:
    print("Installing Python packages:", ", ".join(_missing))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *_missing],
        check=True,
    )
else:
    print("Python prerequisites already satisfied.")

# 2. lz4 CLI -- used to (de)compress the .tar.lz4 archives.
if shutil.which("lz4") is None:
    print("Installing lz4 ...")
    if sys.platform == "darwin":
        subprocess.run(["brew", "install", "lz4"], check=False)
    else:  # Debian/Ubuntu (Colab and most Linux hosts)
        sudo = ["sudo"] if shutil.which("sudo") else []
        subprocess.run([*sudo, "apt-get", "-qq", "update"], check=False)
        subprocess.run([*sudo, "apt-get", "-qq", "install", "-y", "lz4"], check=False)
    if shutil.which("lz4") is None:
        print("WARNING: could not install 'lz4' automatically -- install it manually "
              "(e.g. `apt-get install lz4` or `brew install lz4`).")


# 3. dcmqi -- provides `itkimage2segimage`.
#
# We use the PyPI `dcmqi` package (https://github.com/ImagingDataCommons/
# dcmqi-python-distributions), which bundles the compiled dcmqi CLI binaries.
# NOTE: that package's version is INDEPENDENT of the dcmqi release it ships --
# package 0.4.1 bundles dcmqi binaries v1.5.5 (the first dcmqi with
# `--segmentationType labelmap`, matching the Docker image). So we pin the
# package version, not "1.5.5" (which does not exist on PyPI).
def _find_itkimage2segimage() -> str | None:
    """Return the path to the itkimage2segimage executable, or None.

    Checks PATH first (Docker image / normal installs put the console-script on
    it), then falls back to the binary bundled inside the installed `dcmqi`
    package -- whose console-script shim may land in a directory that is not on
    PATH (e.g. ~/.local/bin on Colab).
    """
    found = shutil.which("itkimage2segimage")
    if found:
        return found
    try:
        from importlib.metadata import PackageNotFoundError, distribution
        for f in distribution("dcmqi").files or []:
            parts = Path(str(f)).parts
            if parts[:2] == ("dcmqi", "bin") and Path(parts[-1]).stem == "itkimage2segimage":
                binary = Path(f.locate()).resolve()
                # Make it (and its siblings: segimage2itkimage, etc.) discoverable.
                os.environ["PATH"] = str(binary.parent) + os.pathsep + os.environ["PATH"]
                return str(binary)
    except (PackageNotFoundError, OSError, IndexError):
        pass
    return None


if _find_itkimage2segimage() is None:
    print("Installing dcmqi (PyPI package 0.4.1 = dcmqi binaries v1.5.5) ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "dcmqi==0.4.1"],
        check=True,
    )

_itk = _find_itkimage2segimage()
if _itk is None:
    print("WARNING: could not install/locate dcmqi -- try `pip install dcmqi`, "
          "or download it from https://github.com/QIICR/dcmqi/releases and put "
          "its bin/ on PATH.")
else:
    print(f"dcmqi itkimage2segimage: {_itk}")

# 4. SNOMED.py -- MOOSE's label->SNOMED mapping module, loaded by a later cell.
#    The WDL wget's this alongside the notebook; fetch it here for standalone runs.
if not Path("SNOMED.py").exists():
    _snomed_url = "https://raw.githubusercontent.com/ENHANCE-PET/MOOSE/main/moosez/mappings/SNOMED.py"
    print(f"Downloading SNOMED.py from {_snomed_url}")
    urllib.request.urlretrieve(_snomed_url, "SNOMED.py")

print("Prerequisite setup complete.")


## Imports

In [ ]:
import json
import os
import re
import shutil
import subprocess
import time
import traceback
from pathlib import Path

import nibabel as nib
import numpy as np
from idc_index.index import IDCClient


## Parameters

Tagged `parameters` so papermill can override via `-p segmentationArchivePath <path>`.

In [ ]:
# Path to the lz4-compressed tar produced by Task 1 (mooseInferenceNotebook)
segmentationArchivePath = "moose_segmentations.tar.lz4"

# --- Optional: copy DICOM-SEG (.dcm) to a GCS bucket ---
# Default off for the IDC pipeline. When set, the per-series uncompressed .dcm
# SEG files are uploaded to this GCS prefix (preserving the <SeriesInstanceUID>/
# layout). On Terra the upload is authenticated with the VM's pet/proxy service
# account via Application Default Credentials -- so NO secret/key is passed as a
# workflow input. Grant that service account roles/storage.objectAdmin on the
# bucket instead.
#
# GCS prefix, e.g.:  gs://idc-not-a-challenge/kyle/
dicomSegBucketUri = ""   # empty = skip upload

# --- Optional: import DICOM-SEG from the bucket above into a Healthcare API DICOM store ---
# Default off for the IDC pipeline. When set, triggers a dicomStores.import job
# that pulls the .dcm files just uploaded to dicomSegBucketUri directly into this
# dicomStore -- this can be a different GCP project than the one running this
# workflow, the resource name fully qualifies the target. Requires
# dicomSegBucketUri to also be set. On Terra the request is authenticated with
# the VM's pet/proxy service account via Application Default Credentials -- so
# NO secret/key is passed as a workflow input. Grant that service account
# roles/healthcare.dicomEditor on the dataset (in whichever project hosts it),
# AND grant the Cloud Healthcare API service agent in that project
# roles/storage.objectViewer on the source bucket (the import job reads from
# GCS server-side, it isn't pushed).
#
# Full dicomStore resource name, e.g.:
#   projects/PROJECT/locations/LOCATION/datasets/DATASET/dicomStores/STORE
dicomStoreImportUri = ""   # empty = skip import

# --- Optional: combine with Task 1 (mooseInferenceNotebook)'s usage metrics ---
# Path to the CSV usage-metrics file produced by Task 1 (only meaningful when
# this notebook is run as Task 2 of the twoVM WDL pipeline, which passes it in
# via papermill). When run standalone -- e.g. from the "Open in Colab" badge
# above, or interactively in Jupyter -- this stays empty and the merge is
# skipped entirely; this notebook never requires Task 1's metrics to run.
# When set, it's merged with this task's own metrics into a combined
# moose_UsageMetrics.csv (see "Write usage metrics" below).
# Empty = skip merge, write post-process-only metrics under the same filename.
inferenceUsageMetricsCsvPath = ""


## Filename utilities and display colors

SNOMED codes themselves come from moosez's own `moosez.mappings.SNOMED` mapping, attached
to each label by the inference notebook and bundled into `moose_organ_indices.json` -- this
notebook does not maintain a separate copy.

In [ ]:
# Regex to extract moosez model identifier from segmentation filenames.
# e.g. "clin_CT_organs_segmentation_CT_foo.nii.gz" -> "clin_ct_organs"
_MOOSE_SEG_FILENAME_RE = re.compile(
    r"^((?:clin|preclin)_[A-Z0-9_-]+?_[a-z][a-z0-9_]*)_segmentation_"
)


def model_from_filename(filename: str) -> str:
    m = _MOOSE_SEG_FILENAME_RE.match(filename)
    return m.group(1).lower() if m else ""


def segment_color(label_id: int) -> list:
    """Visually distinct [R, G, B] display color for a label integer.

    moosez doesn't ship colors for any structure, so every segment gets one of
    these rather than a flat, indistinguishable gray.
    """
    _PALETTE = [
        [255,  85,   0], [  0, 128, 255], [  0, 200,  80], [255, 218,   0],
        [220,   0, 220], [  0, 220, 220], [255, 128,   0], [100, 100, 255],
        [200,   0, 100], [  0, 180, 140], [128, 255,   0], [255,   0, 128],
        [  0, 255, 180], [160,  80, 255], [255, 200, 100], [ 80, 220,  80],
        [255,  50, 150], [ 50, 200, 255], [180, 255,  50], [100, 200, 200],
    ]
    return _PALETTE[(label_id - 1) % len(_PALETTE)]


## Extract the segmentation archive

In [ ]:
EXTRACT_DIR = Path("/tmp/moose_extract")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

subprocess.run(
    f"lz4 -d -c {segmentationArchivePath} | tar -xf - -C {EXTRACT_DIR}",
    shell=True,
    check=True,
)

top_dirs = [p for p in EXTRACT_DIR.iterdir() if p.is_dir()]
if len(top_dirs) == 1:
    MOOSE_ROOT = top_dirs[0]
else:
    MOOSE_ROOT = EXTRACT_DIR

# Load organ indices bundled by the inference step.
# organ_indices: {model_name: {label_int: {"name": organ_name, "snomed": {...} or None}}}
_indices_path = MOOSE_ROOT / "moose_organ_indices.json"
if _indices_path.exists():
    _raw = json.loads(_indices_path.read_text())
    organ_indices = {model: {int(k): v for k, v in labels.items()}
                     for model, labels in _raw.items()}
    print(f"Loaded organ indices for {len(organ_indices)} model(s): {list(organ_indices)}")
else:
    organ_indices = {}
    print("WARNING: moose_organ_indices.json not found — unknown segments will use generic codes")

series_dirs = sorted([p for p in MOOSE_ROOT.iterdir() if p.is_dir()])
print(f"Found {len(series_dirs)} series in {MOOSE_ROOT}")
for p in series_dirs:
    print(f"  {p.name}")


## Helpers

In [ ]:
idc_client = IDCClient()

MODEL_DIR_RE = re.compile(r"^moosez-(?P<model>.+?)-\d{4}[-_]?\d{2}[-_]?\d{2}[-_T]?\d{2}[-_:]?\d{2}[-_:]?\d{2}$")

_ANATOMICAL_STRUCTURE = {
    "CodingSchemeDesignator": "SCT",
    "CodeValue": "123037004",
    "CodeMeaning": "Anatomical Structure",
}
_GENERIC_TYPE = {
    "CodingSchemeDesignator": "SCT",
    "CodeValue": "246183007",
    "CodeMeaning": "Body structure",
}


def organ_entry(model: str, label_id: int) -> dict:
    """{"name": moosez label, "snomed": {"ID", "name"} or None} for a label.

    organ_indices comes from moose_organ_indices.json, bundled by the inference
    notebook with each label's moosez.mappings.SNOMED entry already attached --
    the SNOMED ID/name here is exactly what moosez itself ships.
    """
    return organ_indices.get(model, {}).get(label_id) or {"name": f"segment_{label_id}", "snomed": None}


def download_dicom(uid: str, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    idc_client.download_from_selection(
        downloadDir=str(dest),
        seriesInstanceUID=uid,
    )


def find_series_number(dicom_dir: Path) -> str:
    try:
        import pydicom
        for p in dicom_dir.rglob("*.dcm"):
            ds = pydicom.dcmread(str(p), stop_before_pixels=True)
            sn = getattr(ds, "SeriesNumber", None)
            if sn is not None:
                return str(int(sn))
    except Exception:
        pass
    return "1"


def build_dcmqi_config(model: str, labels: list, series_number: str) -> dict:
    segments = []
    for label_id in labels:
        info = organ_entry(model, int(label_id))
        snomed = info.get("snomed")
        if snomed:
            display_name = snomed["name"]
            seg_type = {
                "CodingSchemeDesignator": "SCT",
                "CodeValue": snomed["ID"],
                "CodeMeaning": snomed["name"],
            }
        else:
            display_name = info["name"]
            seg_type = _GENERIC_TYPE

        segments.append({
            "labelID": int(label_id),
            "SegmentDescription": display_name,
            "SegmentLabel": display_name,
            "SegmentAlgorithmType": "AUTOMATIC",
            "SegmentAlgorithmName": "moosez",
            "SegmentedPropertyCategoryCodeSequence": _ANATOMICAL_STRUCTURE,
            "SegmentedPropertyTypeCodeSequence": seg_type,
            "recommendedDisplayRGBValue": segment_color(int(label_id)),
        })

    return {
        "ContentCreatorName": "MOOSE^CloudSegmentator",
        "ClinicalTrialSeriesID": "Session1",
        "ClinicalTrialTimePointID": "1",
        "SeriesDescription": f"MOOSE ({model}) Segmentation",
        "SeriesNumber": str(int(series_number) * 100 + 1) if series_number.isdigit() else "100",
        "InstanceNumber": "1",
        "BodyPartExamined": "",
        "segmentationType": "LABELMAP",
        "segmentAttributes": [segments],
        "ContentLabel": "SEGMENTATION",
        "ContentDescription": f"moosez {model} multi-label segmentation",
        "ClinicalTrialCoordinatingCenterName": "",
    }


def unique_nonzero_labels(nifti_path: Path) -> list:
    arr = np.asanyarray(nib.load(str(nifti_path)).dataobj)
    return sorted(int(v) for v in np.unique(arr) if v != 0)


## Generate DICOM-SEG per series/model

In [ ]:
DICOM_DIR = Path("/tmp/dicom")
DICOM_SEG_DIR = Path("/tmp/moose_dicom_seg")
CONFIG_DIR = Path("/tmp/moose_configs")
for d in (DICOM_DIR, DICOM_SEG_DIR, CONFIG_DIR):
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True)

dicom_seg_errors = []
usage_metrics = {"series": {}}

for series_dir in series_dirs:
    uid = series_dir.name
    print(f"\n=== Processing {uid} ===")
    series_start = time.time()

    dicom_dest = DICOM_DIR / uid
    try:
        t0 = time.time()
        download_dicom(uid, dicom_dest)
        usage_metrics["series"].setdefault(uid, {})["download_s"] = round(time.time() - t0, 1)
    except Exception as exc:
        msg = f"{uid}: download failed: {exc}\n{traceback.format_exc()}"
        dicom_seg_errors.append(msg)
        print(f"ERROR: {msg}")
        continue

    series_number = find_series_number(dicom_dest)

    seg_out_dir = DICOM_SEG_DIR / uid
    seg_out_dir.mkdir(parents=True, exist_ok=True)

    model_dirs = [p for p in series_dir.iterdir() if p.is_dir() and MODEL_DIR_RE.match(p.name)]
    synthetic_model_dirs = False
    if not model_dirs:
        flat_seg_files = sorted(series_dir.rglob("*.nii.gz"))
        if flat_seg_files:
            synthetic_model_dirs = True
            model_dirs = [series_dir]
            print(f"INFO: {uid}: using legacy flat segmentation layout ({len(flat_seg_files)} files)")
        else:
            msg = f"{uid}: no model directories or .nii.gz segmentations found in {series_dir}"
            dicom_seg_errors.append(msg)
            print(f"WARN: {msg}")
            continue

    for model_dir in model_dirs:
        if synthetic_model_dirs:
            model = "legacy"
            seg_files = sorted(model_dir.rglob("*.nii.gz"))
        else:
            m = MODEL_DIR_RE.match(model_dir.name)
            model = m.group("model") if m else model_dir.name
            seg_files = list((model_dir / "segmentations").glob("*.nii.gz")) \
                if (model_dir / "segmentations").exists() \
                else list(model_dir.rglob("*.nii.gz"))

        if not seg_files:
            # Only count as an error if this is a genuine moosez model directory
            # (matches MODEL_DIR_RE). Support directories like "stats" are silently skipped.
            if not synthetic_model_dirs and not MODEL_DIR_RE.match(model_dir.name):
                print(f"  INFO: {model_dir.name}: no segmentations found, skipping non-model directory")
            else:
                msg = f"{uid}/{model}: no .nii.gz segmentations under {model_dir}"
                dicom_seg_errors.append(msg)
                print(f"WARN: {msg}")
            continue

        for seg_idx, seg_file in enumerate(seg_files):
            try:
                labels = unique_nonzero_labels(seg_file)
                if not labels:
                    print(f"  {model}/{seg_file.name}: empty mask, skipping")
                    continue

                effective_model = model
                if effective_model not in organ_indices:
                    fn_model = model_from_filename(seg_file.name)
                    if fn_model:
                        effective_model = fn_model

                cfg = build_dcmqi_config(effective_model, labels, series_number)
                cfg_path = CONFIG_DIR / f"{uid}_{effective_model}_{seg_idx}.json"
                cfg_path.write_text(json.dumps(cfg, indent=2))

                out_dcm = seg_out_dir / f"{effective_model}_{seg_idx}.dcm"
                t0 = time.time()
                result = subprocess.run(
                    [
                        "itkimage2segimage",
                        "--inputImageList", str(seg_file),
                        "--inputDICOMDirectory", str(dicom_dest),
                        "--outputDICOM", str(out_dcm),
                        "--inputMetadata", str(cfg_path),
                        "--segmentationType", "labelmap",
                        # Preserve the moosez label values as DICOM Segment Numbers
                        # instead of letting dcmqi renumber segments 1..N. Keeps the
                        # SEG segment numbering aligned with the source labelmap.
                        "--useLabelIDAsSegmentNumber",
                        # Skip empty slices (default in dcmqi 1.5.5; the bare --skip
                        # boolean from older dcmqi now requires an int value).
                        "--skip", "1",
                    ],
                    capture_output=True,
                    text=True,
                )
                elapsed = round(time.time() - t0, 1)

                if result.returncode != 0 or not out_dcm.exists():
                    msg = (
                        f"{uid}/{model}/{seg_file.name}: itkimage2segimage failed "
                        f"(rc={result.returncode})\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}"
                    )
                    dicom_seg_errors.append(msg)
                    print(f"  ERROR: {msg}")
                else:
                    usage_metrics["series"].setdefault(uid, {}).setdefault("models", {})[effective_model] = elapsed
                    print(f"  {effective_model}: wrote {out_dcm.name} ({elapsed}s, {len(labels)} segments)")

            except Exception as exc:
                msg = f"{uid}/{model}/{seg_file.name}: {traceback.format_exc()}"
                dicom_seg_errors.append(msg)
                print(f"  ERROR: {exc}")

    usage_metrics["series"].setdefault(uid, {})["total_s"] = round(time.time() - series_start, 1)
    shutil.rmtree(dicom_dest, ignore_errors=True)

if dicom_seg_errors:
    Path("dicom_seg_error_file.txt").write_text("\n\n".join(dicom_seg_errors))


## Package DICOM-SEG outputs

In [ ]:
def compress_dir(src_dir: Path, out_file: str) -> None:
    cmd = f"tar -cf - -C {src_dir.parent} {src_dir.name} | lz4 > {out_file}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Compression failed: {result.stderr}")
    size_mb = Path(out_file).stat().st_size / (1024 ** 2)
    print(f"Created {out_file} ({size_mb:.1f} MB)")

compress_dir(DICOM_SEG_DIR, "moose_dicom_seg.tar.lz4")

## (Optional) Copy DICOM-SEG to a GCS bucket

Default off. When `dicomSegBucketUri` is set, the converted per-series `.dcm`
SEG files are uploaded to that GCS prefix, preserving the
`<SeriesInstanceUID>/<file>.dcm` layout.

Authentication uses **Application Default Credentials** — on Terra that is the
VM's pet/proxy service account, so no key or secret is passed as a workflow
input. The account needs `roles/storage.objectAdmin` (or at least
`objectCreator`) on the destination bucket. Note this only deposits files in the
bucket; loading them into a DICOM store for OHIF is a separate import step.

In [ ]:
def upload_dicom_seg_to_bucket() -> None:
    if not dicomSegBucketUri:
        print("Bucket upload skipped (set dicomSegBucketUri to enable)")
        return
    if not dicomSegBucketUri.startswith("gs://"):
        raise ValueError(f"dicomSegBucketUri must start with gs:// , got {dicomSegBucketUri!r}")

    from google.cloud import storage

    # gs://bucket/prefix -> (bucket, prefix)
    bucket_name, _, prefix = dicomSegBucketUri[len("gs://"):].partition("/")
    prefix = prefix.strip("/")

    # ADC: on Terra this is the VM's pet/proxy service account (no secret).
    # Needs roles/storage.objectAdmin on the bucket.
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    print(f"Uploading SEGs to gs://{bucket_name}/{prefix}")

    uploaded = 0
    errors = []
    for series_seg_dir in sorted(p for p in DICOM_SEG_DIR.iterdir() if p.is_dir()):
        for dcm in sorted(series_seg_dir.glob("*.dcm")):
            # Preserve <SeriesInstanceUID>/<file>.dcm under the prefix.
            rel = f"{series_seg_dir.name}/{dcm.name}"
            blob_name = f"{prefix}/{rel}" if prefix else rel
            try:
                bucket.blob(blob_name).upload_from_filename(str(dcm))
                uploaded += 1
                print(f"  uploaded gs://{bucket_name}/{blob_name}")
            except Exception as exc:
                # Capture the HTTP response body when present so the cause is
                # recorded in the raised message (the only thing papermill echoes).
                detail = str(exc)
                resp = getattr(exc, "response", None)
                if resp is not None:
                    detail += f"\nHTTP {resp.status_code} response body:\n{resp.text}"
                msg = f"{rel}: upload failed: {detail}"
                errors.append(msg)
                print(f"  ERROR: {msg}")

    if errors:
        Path("dicom_bucket_error_file.txt").write_text("\n\n".join(errors))
    print(f"Bucket upload: {uploaded} file(s) uploaded, {len(errors)} failed")
    # Hard-fail only if nothing uploaded; partial success keeps the archive output.
    if errors and uploaded == 0:
        raise RuntimeError(
            f"Bucket upload failed for all files (dest: {dicomSegBucketUri}):\n\n"
            + "\n\n".join(errors)
        )


upload_dicom_seg_to_bucket()


## (Optional) Import DICOM-SEG into a Healthcare API DICOM store

Default off. When `dicomStoreImportUri` is set, this triggers a
`dicomStores.import` job that pulls the `.dcm` files just uploaded to
`dicomSegBucketUri` directly into that dicomStore. The target dicomStore
can live in a different GCP project than the one running this workflow --
the resource name fully qualifies project/location/dataset/store, and
Healthcare API authorization is IAM-based on that resource rather than on
the project of the calling credentials.

Authentication uses **Application Default Credentials** — on Terra that is
the VM's pet/proxy service account, so no key or secret is passed as a
workflow input. Grant that service account `roles/healthcare.dicomEditor`
on the dataset in whichever project hosts it. Unlike a direct push, the
import job reads from GCS server-side, so the Cloud Healthcare API service
agent in that same project also needs `roles/storage.objectViewer` on the
source bucket. The import runs as a long-running operation, polled here
until it completes or the timeout is hit; dcmqi preserves the source
`StudyInstanceUID`, so once imported the SEGs attach to the existing study
and render as overlays in OHIF — the source images must already be in the
store.

In [ ]:
def import_dicom_seg_into_dicom_store() -> None:
    if not dicomStoreImportUri:
        print("DICOM store import skipped (set dicomStoreImportUri to enable)")
        return
    if not dicomSegBucketUri:
        raise ValueError(
            "dicomStoreImportUri requires dicomSegBucketUri to also be set -- "
            "the import job reads the DICOM-SEG files from that GCS location."
        )

    import time

    from google.auth import default as google_auth_default
    from google.auth.transport.requests import AuthorizedSession
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry

    # Application Default Credentials: on Terra this is the VM's pet/proxy
    # service account, so no secret is passed in. Needs roles/healthcare.dicomEditor
    # on the target dataset -- which may be in a different project than this VM --
    # and the Healthcare API service agent in that project needs
    # roles/storage.objectViewer on the source bucket (the import reads from GCS
    # server-side rather than receiving a push).
    creds, project = google_auth_default(
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )
    store_name = dicomStoreImportUri.strip("/")
    gcs_uri = dicomSegBucketUri.rstrip("/") + "/**"
    sa_email = getattr(creds, "service_account_email", "unknown")
    print(f"Import target: {store_name}")
    print(f"Import source: {gcs_uri}")
    print(f"ADC project: {project}, service account: {sa_email}")

    # Retry transient Healthcare API errors (429 rate limit, 5xx) with
    # exponential backoff; AuthorizedSession refreshes the bearer token on demand.
    session = AuthorizedSession(creds)
    retry = Retry(
        total=5,
        backoff_factor=2,  # 2, 4, 8, 16, 32 seconds
        status_forcelist=[429, 500, 502, 503],
        allowed_methods=frozenset(["GET", "POST"]),
    )
    session.mount("https://", HTTPAdapter(max_retries=retry))

    base_url = "https://healthcare.googleapis.com/v1"
    try:
        resp = session.post(
            f"{base_url}/{store_name}:import",
            json={"gcsSource": {"uri": gcs_uri}},
            timeout=60,
        )
        resp.raise_for_status()
        operation_name = resp.json()["name"]
        print(f"Import operation started: {operation_name}")

        poll_interval_seconds = 10
        timeout_seconds = 1800
        deadline = time.monotonic() + timeout_seconds
        operation = {}
        while time.monotonic() < deadline:
            op_resp = session.get(f"{base_url}/{operation_name}", timeout=60)
            op_resp.raise_for_status()
            operation = op_resp.json()
            if operation.get("done"):
                break
            time.sleep(poll_interval_seconds)
        else:
            raise RuntimeError(
                f"DICOM store import timed out after {timeout_seconds}s "
                f"(operation: {operation_name}, still running -- check the "
                f"Healthcare API console for {store_name})"
            )

        if "error" in operation:
            raise RuntimeError(
                f"DICOM store import failed: {operation['error'].get('message', operation['error'])}"
            )

        print(f"Import complete -> {store_name} ({operation.get('response', {})})")
    except Exception as exc:
        # Capture the HTTP response body when present (e.g. the 403/400 detail
        # from the Healthcare API) so the real cause is recorded, not just the
        # raised exception message that papermill echoes to the Terra console.
        detail = str(exc)
        resp = getattr(exc, "response", None)
        if resp is not None:
            detail += f"\nHTTP {resp.status_code} response body:\n{resp.text}"
        msg = f"Import failed (SA: {sa_email}): {detail}"
        Path("dicom_import_error_file.txt").write_text(msg)
        raise RuntimeError(msg) from exc


import_dicom_seg_into_dicom_store()


## Write usage metrics

In [ ]:
import csv

# ── Flatten per-(series, model) timing into a CSV for easy analysis ────────
# series_* columns are repeated across every model row for that series;
# model_* varies per (series, model) row.
csv_path = Path("moose_postprocess_UsageMetrics.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "SeriesInstanceUID", "model",
        "model_itkimage2segimage_s",
        "series_download_s", "series_total_s",
    ])
    for uid, series_metrics in usage_metrics["series"].items():
        model_times = series_metrics.get("models", {})
        for model in (list(model_times) or [""]):
            writer.writerow([
                uid,
                model,
                model_times.get(model, ""),
                series_metrics.get("download_s", ""),
                series_metrics.get("total_s", ""),
            ])

print(json.dumps(usage_metrics, indent=2))
print(f"Usage metrics written ({csv_path.name})")


## (Optional) Merge with Task 1 usage metrics and upload to GCS

Default off (no-op if `inferenceUsageMetricsCsvPath` isn't set -- which is
always the case for a standalone/Colab run). When this notebook is run as Task
2 of the twoVM WDL pipeline, that parameter is populated with Task 1's
`moose_inference_UsageMetrics.csv`, localized onto this VM by Cromwell. When
present, it's merged with this task's own usage metrics above into a combined
`moose_UsageMetrics.csv`, then uploaded to a `_metrics/` subfolder under
`dicomSegBucketUri` -- alongside, not inside, the per-series DICOM-SEG
folders.

Both phases have a `series_download_s`-shaped column with different meanings
(Task 1 downloads DICOM to convert to NIfTI; Task 2 re-downloads it later
purely as an `itkimage2segimage` reference directory), so the merged CSV
prefixes every column with `inference_`/`postprocess_` to disambiguate without
losing either value.

Metrics are supplementary telemetry, not the core deliverable, so the upload
step warns and continues on failure rather than raising -- unlike
`upload_dicom_seg_to_bucket()` above, this never fails the task.

In [ ]:
def build_postprocess_metrics_rows() -> dict:
    """Re-flatten this notebook's own usage_metrics into per-(series, model)
    rows, matching the column shape already written to
    moose_postprocess_UsageMetrics.csv above."""
    rows = {}
    for uid, series_metrics in usage_metrics["series"].items():
        model_times = series_metrics.get("models", {})
        for model in (list(model_times) or [""]):
            rows[(uid, model)] = {
                "model_itkimage2segimage_s": model_times.get(model, ""),
                "series_download_s": series_metrics.get("download_s", ""),
                "series_total_s": series_metrics.get("total_s", ""),
            }
    return rows


def load_inference_metrics_rows() -> dict:
    """Load Task 1's flattened per-(series, model) CSV rows. Returns {} if not
    configured or unreadable -- the merge degrades gracefully to
    post-process-only output rather than failing the notebook. The path is
    empty by default, so a standalone/Colab run always takes this branch."""
    if not inferenceUsageMetricsCsvPath:
        print("Inference metrics merge skipped (inferenceUsageMetricsCsvPath not set)")
        return {}

    rows = {}
    try:
        with open(inferenceUsageMetricsCsvPath, newline="") as f:
            for row in csv.DictReader(f):
                key = (row.get("SeriesInstanceUID", ""), row.get("model", ""))
                rows[key] = {
                    k: v for k, v in row.items()
                    if k not in ("SeriesInstanceUID", "model")
                }
    except (OSError, csv.Error) as exc:
        print(f"WARNING: failed to read inference usage metrics CSV "
              f"({inferenceUsageMetricsCsvPath}): {exc}")
        return {}

    return rows


inference_rows = load_inference_metrics_rows()
postprocess_rows = build_postprocess_metrics_rows()

# ── Merge per-(series, model) rows from both phases ────────────────────────
# Column collision note: both phases have a "series_download_s" column with
# DIFFERENT meanings (Task 1 downloads DICOM to convert to NIfTI; Task 2
# re-downloads DICOM later purely as an itkimage2segimage reference dir).
# Layer inference_/postprocess_ prefixes on top of each phase's own
# already-finalized column names to disambiguate without inventing new names
# or losing information.
INFERENCE_COLUMNS = [
    "series_checkpoint_restored",
    "series_download_s", "series_download_dcm_files", "series_download_mb",
    "series_rows", "series_cols", "series_slices", "series_total_pixels",
    "series_dcm2niix_s", "series_moose_s",
    "model_inference_s", "model_stats_s",
    "run_gpu_name", "run_gpu_total_mb", "run_total_elapsed_s",
]
POSTPROCESS_COLUMNS = [
    "model_itkimage2segimage_s",
    "series_download_s", "series_total_s",
]

combined_csv_path = Path("moose_UsageMetrics.csv")
all_keys = sorted(set(inference_rows) | set(postprocess_rows))
with open(combined_csv_path, "w", newline="") as f:
    fieldnames = (
        ["SeriesInstanceUID", "model"]
        + [f"inference_{c}" for c in INFERENCE_COLUMNS]
        + [f"postprocess_{c}" for c in POSTPROCESS_COLUMNS]
    )
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for uid, model in all_keys:
        inf = inference_rows.get((uid, model), {})
        post = postprocess_rows.get((uid, model), {})
        row = {"SeriesInstanceUID": uid, "model": model}
        for c in INFERENCE_COLUMNS:
            row[f"inference_{c}"] = inf.get(c, "")
        for c in POSTPROCESS_COLUMNS:
            row[f"postprocess_{c}"] = post.get(c, "")
        writer.writerow(row)

print(f"Combined usage metrics written ({combined_csv_path.name})")


def upload_usage_metrics_to_bucket() -> None:
    """Upload the combined usage-metrics CSV to a _metrics/ subfolder under
    the same bucket+prefix as the DICOM-SEG uploads. Metrics are supplementary
    telemetry, not the core deliverable -- never raise, only warn."""
    if not dicomSegBucketUri:
        print("Usage metrics bucket upload skipped (set dicomSegBucketUri to enable)")
        return
    if not dicomSegBucketUri.startswith("gs://"):
        print(f"WARNING: dicomSegBucketUri does not start with gs:// "
              f"({dicomSegBucketUri!r}), skipping usage metrics upload")
        return

    from google.cloud import storage

    bucket_name, _, prefix = dicomSegBucketUri[len("gs://"):].partition("/")
    prefix = prefix.strip("/")

    client = storage.Client()
    bucket = client.bucket(bucket_name)
    metrics_prefix = f"{prefix}/_metrics" if prefix else "_metrics"
    blob_name = f"{metrics_prefix}/{combined_csv_path.name}"
    print(f"Uploading combined usage metrics to gs://{bucket_name}/{blob_name}")

    try:
        bucket.blob(blob_name).upload_from_filename(str(combined_csv_path))
        print(f"  uploaded gs://{bucket_name}/{blob_name}")
    except Exception as exc:
        detail = str(exc)
        resp = getattr(exc, "response", None)
        if resp is not None:
            detail += f"\nHTTP {resp.status_code} response body:\n{resp.text}"
        msg = f"{combined_csv_path.name}: upload failed: {detail}"
        Path("metrics_bucket_error_file.txt").write_text(msg)
        print(f"  WARNING: {msg}")
    # Intentionally never raises -- metrics are supplementary, not the core
    # deliverable. dicomSegArchive output is unaffected either way.


upload_usage_metrics_to_bucket()


## Summary

In [ ]:
print("=" * 60)
print("MOOSE Post-Process Summary")
print("=" * 60)
print(f"  Series processed : {len(series_dirs)}")
print(f"  Errors logged    : {len(dicom_seg_errors)}")
print("=" * 60)

if len(dicom_seg_errors) and len(dicom_seg_errors) >= len(series_dirs):
    raise RuntimeError("DICOM-SEG generation failed for all series - see dicom_seg_error_file.txt")